# Semi-explicit Solution to Stochastic Differential Games on Graphs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
from math import isclose
import torch
import sys
import networkx as nx
import os
import scipy
import datetime
import sklearn
from sklearn.linear_model import LinearRegression

# Check the output directory
assert os.path.isdir('plots'), 'No output directory called plots exist!'

Since this piece of code does not involve deep learning, it always runs on the CPU.

In [ ]:
device = torch.device('cpu')
print("Running on the CPU!")

## Model Settings and Parameters

There are $N$ players in the game with the dynamics given by
$$dX^i_t = \left[a\left(\frac{1}{\sqrt{d_{v_i}}}\sum_{j:v_j\sim v_i}\frac{1}{\sqrt{d_{v_j}}}X^j_t - X^i_t\right) + \alpha^i_t\right]\,dt + \sigma\,dW_t^i$$
where $a$ is the mean-reverting parameter, $d_{v_i}$ is the degree of vertex $v_i$ in the graph, $\alpha^i_t$ is the control of player $i$ at time $t$ and $(W^0_t,...,W^{N-1}_t)$ is an $N$-dimensional Brownian motion. The $\sim$ relationship between two vertices $v_i,v_j$ in the graph means that there exists an edge connecting $v_i$ to $v_j$. The running cost of player $i$ in this problem is set as
$$f^i(X_t,\alpha^i_t) = \frac{1}{2}(\alpha^i_t)^2 - q\alpha^i_t\left(\frac{1}{\sqrt{d_{v_i}}}\sum_{j:v_j\sim v_i}\frac{1}{\sqrt{d_{v_j}}}X^j_t - X^i_t\right) + \frac{\varepsilon}{2}\left(\frac{1}{\sqrt{d_{v_i}}}\sum_{j:v_j\sim v_i}\frac{1}{\sqrt{d_{v_j}}}X^j_t - X^i_t\right)^2$$
the terminal cost of player $i$ is set as
$$g^i(X_T) = \frac{c}{2}\left(\frac{1}{\sqrt{d_{v_i}}}\sum_{j:v_j\sim v_i}\frac{1}{\sqrt{d_{v_j}}}X^j_T - X^i_T\right)^2$$
with $a,\sigma,q,\varepsilon,c>0$ as parameters. We want to derive the Markovian Nash equilibrium where each player minimizes its expected cost
$$J^i(\alpha) = \mathbb{E}\left[\int_0^Tf^i(X_t,\alpha^i_t)\,dt + g^i(X_T)\right]$$
with time horizon $[0,T]$.

In [ ]:
'''
Model parameters
'''
N_PLAYER = 50
MOD_a = 0.1
MOD_sigma = 0.5
MOD_q = 0.5
MOD_eps = 1.0
MOD_c = 1.0
MOD_T = 1.0
MOD_DELTA = 1.0 # Initial state value U(-delta,delta)

The graph structure is encoded in the graph Laplacian $L$. Here we require the graph to be simple, connected and undirected, but not necessarily transitive. The graph Laplacian is defined as follows:
$$L\in\mathbb{R}^{N\times N},L_{ij} = \begin{cases}1 & i=j\\ -\frac{1}{\sqrt{d_{v_i}d_{v_j}}} & v_i\sim v_j\\ 0 & \text{else}\end{cases}$$

In [ ]:
'''
The following function returns the Laplacian of some commonly appearing graphs
sparsity index only used when the generated graph is random
it's organized as |E|/\binom{|V|}{2}, the number of edges over the number of all possible potential edges
'''
graph_name_lst = ['complete','star','cycle','petersen','hypercube','circulant',
                  'complete bipartite','RSG','Ramanujan']

def set_graph_LAP(name,graph_name_lst,sparsity_index = 0.125, degree = 6):
    assert (name in graph_name_lst), 'Graph name error!'
        
    if name == 'complete':
        G = nx.complete_graph(range(N_PLAYER))
        LAP = nx.normalized_laplacian_matrix(G).toarray()
    elif name == 'star':
        G = nx.star_graph(range(N_PLAYER))
        LAP = nx.normalized_laplacian_matrix(G).toarray()
    elif name == 'cycle':
        G = nx.cycle_graph(range(N_PLAYER))
        LAP = nx.normalized_laplacian_matrix(G).toarray() 
    elif name == 'petersen':
        assert N_PLAYER == 10, 'Error! Petersen graph must have 10 vertices!'
        G = nx.petersen_graph()
        LAP = nx.normalized_laplacian_matrix(G).toarray() 
    elif name == 'hypercube':
        k = int(np.round(np.log(N_PLAYER) / np.log(2)))
        assert 2 ** k == N_PLAYER, 'Error! N is not a power of 2!'
        G = nx.hypercube_graph(k)
        LAP = nx.normalized_laplacian_matrix(G).toarray() 
    elif name == 'circulant':
        G = nx.circulant_graph(N_PLAYER,offsets = [11])
        LAP = nx.normalized_laplacian_matrix(G).toarray()
    elif name == 'complete bipartite':
        m = N_PLAYER // 2
        assert 2 * m == N_PLAYER, 'Number of player is odd!'
        G = nx.complete_bipartite_graph(m, m)
        LAP = nx.normalized_laplacian_matrix(G).toarray()
    elif name == 'RSG':
        # Random sparse graph with the indicated sparsity index
        # Generate a complete graph
        G = nx.complete_graph(range(N_PLAYER))
        
        # Get a random spanning tree (uniform) and relabel all the nodes
        G = nx.random_spanning_tree(G)
        G = nx.convert_node_labels_to_integers(G)
        
        # Add edges until it meets the required sparsity
        exp_edge_num = int(np.round(N_PLAYER * (N_PLAYER - 1) / 2 * sparsity_index))
        current_edge_num = N_PLAYER - 1
        while current_edge_num < exp_edge_num:
            # Add a random edge (make sure the graph is simple)
            i = np.random.randint(0,N_PLAYER)
            j = np.random.randint(0,N_PLAYER)
            if i == j or G.has_edge(i,j):
                continue
            else:
                G.add_edge(i,j)
                current_edge_num = current_edge_num + 1   
        # Laplacian
        LAP = nx.normalized_laplacian_matrix(G).toarray()
    elif name == 'Ramanujan':
        G = nx.random_regular_expander_graph(N_PLAYER, d = degree, epsilon = 0, 
                                            max_tries = 1000)
        # Laplacian
        LAP = nx.normalized_laplacian_matrix(G).toarray()
        
    # Save the graph plot
    nx.draw_networkx(G,with_labels = True)
    plt.savefig('plots/graph_structure.jpg',dpi = 600)
    plt.close()
    return LAP

In [ ]:
# Graph structure, LAP (N_PLAYER * N_PLAYER)
graph_name = 'Ramanujan' # What's the name of the graph
LAP = set_graph_LAP(graph_name,graph_name_lst)

# If to build semi-explicit NE for transitive graphs
if (graph_name in ['complete','cycle','petersen','hypercube','circulant','complete bipartite']
    and isclose(MOD_eps,MOD_q ** 2)):
    TRANSITIVE = True
else:
    TRANSITIVE = False

Since we need to numerically simulate the SDE solution with Euler scheme, it's good to specify how we are discretizing the time interval $[0,T]$ and how many samples we are using in Monte Carlo to approximate the expected cost.

In [ ]:
'''
Time discretization
'''
N_STEPS = 50 # Num of time steps in time discretization when plotting
N_STEPS_EXACT = 1000 # Num of time steps in time discretization when numerically solving ODEs

In [ ]:
# N_STEPS_EXACT must be integer multiple of N_STEPS
assert (N_STEPS_EXACT >= N_STEPS and N_STEPS_EXACT % N_STEPS == 0), 'Error! N_STEPS_EXACT is not a multiple of N_STEPS!'

The number of trajectories we take to average when we compute the expected costs.

In [ ]:
'''
Evaluation and test setting
'''
N_SIM_TEST = 10  # Expected cost calculated based on how many samples
N_FIXED_BM = 1 # How many BM trajectories to fix when plotting
assert (N_FIXED_BM <= 10), 'Too many trajectories to plot!'

The error tolerance in FP, indicating when to stop.

In [ ]:
'''
FP stopping criterion
'''
FP_TOL = 1e-4
N_FP_ROUND = 10

Fix a random seed when plotting the trajectories so it's possible to compare the performance of different set of hyperparameters when doing fine tuning.

In [ ]:
'''
Random seeds
'''
TORCH_INIT_SEED = 42 # Seed for initial state in testing
TORCH_PLOT_SEED = 100 # Seed for plotting

## Graph Information

The main information we are using is the degree and neighbors of each vertex in the graph.

In [ ]:
'''
Get the degrees and neighbors of each vertex in the graph
Input: Graph Laplacian (N_PLAYER * N_PLAYER)
Output: degree list (N_PLAYER), a list of list indicating neighbors
'''
def get_graph_info(LAP):
    N_PLAYER = LAP.shape[0]
    
    deg_lst = list() # Degree list
    nb_lst_lst = list() # neighbor list (a list of list)
    
    for vertex_i in range(N_PLAYER):
        LAP_col = LAP[:,vertex_i]
        nb_lst = list()
        deg_count = 0
        for vertex_j in range(N_PLAYER):
            # Do not count itself
            if vertex_i == vertex_j:
                continue
            if not isclose(LAP_col[vertex_j],0):
                # An edge detected
                deg_count = deg_count + 1
                nb_lst.append(vertex_j)
        # Record
        deg_lst.append(deg_count)
        nb_lst_lst.append(nb_lst)
    return deg_lst, nb_lst_lst

Get the degree list and the neighbor list that will be used later. Notice that the neighbor list is a list of list, its $i$-th entry is a list containing all neighbors of player $i$.

In [ ]:
# Get the information of the graph (degree list, neighbor list)
DEG_LST, NB_LST_LST = get_graph_info(LAP)

# Check if graph is connected
assert (0 not in DEG_LST), 'Error! The graph has to be connected!'

## Dynamics

The following function computes the term $\left(\frac{1}{\sqrt{d_{v_i}}}\sum_{j:v_j\sim v_i}\frac{1}{\sqrt{d_{v_j}}}X^j_t - X^i_t\right)$ for player $i$.

In [ ]:
'''
Calculate mean-reverting term for a certain player
Input: X_tensor as 2-dim tensor (N_PLAYER * N_SIM), the index of player
Output: a 2-dim column tensor (N_SIM * 1)
'''

'''
# This is the version for the exact mean reversion in the nbhd
def mean_rev_term(X_tensor,NB_LST_LST,ply_ind):
    # All neighbors of that player
    nb_ply_ind = NB_LST_LST[ply_ind]
    
    # Maintain the dimension as 2-dim (N_SIM * 1)
    X_tensor_nb = X_tensor[nb_ply_ind,:]
    mean_tensor = torch.unsqueeze(torch.mean(X_tensor_nb,dim = 0),1)
    ply_ind_tensor = torch.unsqueeze(X_tensor[ply_ind,:],1)
    return mean_tensor - ply_ind_tensor
'''

# This is the version for the generic model above (square root of degree)
def mean_rev_term(X_tensor,NB_LST_LST,ply_ind):
    # All neighbors of that player and their square root degrees
    nb_ply_ind = NB_LST_LST[ply_ind]
    nb_sqrt_deg = torch.sqrt(torch.Tensor(DEG_LST)[nb_ply_ind]).to(device)
    
    # Maintain the dimension as 2-dim (N_SIM * 1)
    X_tensor_nb = X_tensor[nb_ply_ind,:]
    ply_sqrt_deg = np.sqrt(DEG_LST[ply_ind])
    mean_tensor = 1 / ply_sqrt_deg * torch.unsqueeze(torch.sum(X_tensor_nb / nb_sqrt_deg[:,None],dim = 0),1)
    ply_ind_tensor = torch.unsqueeze(X_tensor[ply_ind,:],1)
    return mean_tensor - ply_ind_tensor

Build up functions in the dynamics of a certain player. Here it's taken as a linear-quadratic game.

In [ ]:
'''
Input: X_tensor as 2-dim tensor (N_PLAYER * N_SIM), the index of the player, model parameters
control of the player as 2-dim tensor (N_SIM * 1)
Output: a 2-dim column tensor (N_SIM * 1)
'''
# Drift
def dynamics_b(X_tensor,ply_ind,NB_LST_LST,MOD_a,ctrl_ply_ind):
    mean_rev = mean_rev_term(X_tensor,NB_LST_LST,ply_ind)
    return MOD_a * mean_rev + ctrl_ply_ind

# Running cost rate 
def dynamics_f(X_tensor,ply_ind,NB_LST_LST,MOD_q,MOD_eps,ctrl_ply_ind):
    mean_rev = mean_rev_term(X_tensor,NB_LST_LST,ply_ind)
    return 0.5 * (ctrl_ply_ind ** 2) - MOD_q * ctrl_ply_ind * mean_rev + 0.5 * MOD_eps * (mean_rev ** 2)

# Terminal cost
def dynamics_g(X_tensor,ply_ind,NB_LST_LST,MOD_c):
    mean_rev = mean_rev_term(X_tensor,NB_LST_LST,ply_ind)
    return 0.5 * MOD_c * (mean_rev ** 2)

## Exact Solution to NE

### Solve Riccati Equation

The exact solution is derived by noticing that in graph LQ game the Markovian Nash equilibrium is attained by the control of player $i$ at time $t$ facing state vector $X_t$ that
$$\hat{\alpha}^i(t,X_t) = -qe_i^TLX_t - e_i^TF^i_tX_t$$
where $F^i:[0,T]\to \mathbb{S}^{N\times N}$ takes value as a symmetric matrix determined by the following matrix-valued Riccati equation with given terminal condition
$$\dot{F}^i_t + \sum_{k=1}^N [-(a+q)L-F^k_t]e_ke_k^TF^i_t + \sum_{k=1}^N F^i_te_ke_k^T[-(a+q)L-F^k_t] + (\varepsilon - q^2)Le_ie_i^TL + F^i_te_ie_i^TF^i_t = 0$$
$$F^i_T = cLe_ie_i^TL$$
we use the method in scipy package to numerically solve the coupled ODE system and get $F^1,...,F^N$.

In [ ]:
'''
The following function calculates the derivative of F^i, flattened as a vector and concatenated.
Input: t as time, F as vec(F^0),...,vec(F^{N-1}) concatenated together (of length N_PLAYER^3), model parameters
Output: the derivative of F (of length N_PLAYER^3)
'''
def Riccati(t,F_vec,MOD_a,MOD_q,LAP,MOD_eps,N_PLAYER,fund_mat_list):
    # Cut vector y into N_PLAYER pieces and reshape into matrices
    F_piece_list = np.array_split(F_vec,N_PLAYER)
    F_mat_list = [np.reshape(F_piece,(N_PLAYER,N_PLAYER)) for F_piece in F_piece_list]
    
    # Calculate derivative
    F_prime_mat_list = list()
    for i in range(N_PLAYER):
        F_prime_mat = np.zeros((N_PLAYER,N_PLAYER))
        # Riccati equation above
        for k in range(N_PLAYER):
            LAP_term = (MOD_a + MOD_q) * LAP + F_mat_list[k]
            F_prime_mat = F_prime_mat + ((LAP_term @ fund_mat_list[k] @ F_mat_list[i]) 
                                         + (F_mat_list[i] @ fund_mat_list[k] @ LAP_term))

        F_prime_mat = F_prime_mat - (MOD_eps - MOD_q ** 2) * (LAP @ fund_mat_list[i] @ LAP)
        F_prime_mat = F_prime_mat - F_mat_list[i] @ fund_mat_list[i] @ F_mat_list[i]
        
        # Append
        F_prime_mat_list.append(F_prime_mat)
    
    # Flatten back into a vector and concatenate
    F_prime_vec = np.concatenate([np.reshape(F_prime_mat,(-1,)) for F_prime_mat in F_prime_mat_list])
    
    return F_prime_vec

In [ ]:
'''
The following function returns standard basis column vector e_i in R^N
Output: a 2-dim column vector of shape N * 1 as nparray
'''
def get_std_basis(i,N):
    std_basis = np.zeros(N)
    std_basis[i] = 1.0
    std_basis = np.reshape(std_basis,(-1,1))
    return std_basis

In [ ]:
'''
The following function returns a list of fundamental matrices with only one non-zero entry.
The nonzero entry is 1 on the diagonal.
Output: a list of fundamental matrices, i.e. e_1e_1^T, e_2e_2^T, ..., e_Ne_N^T
'''
def get_fund_mat_list(N_PLAYER):
    fund_mat_list = list()
    for i in range(N_PLAYER):
        std_basis = get_std_basis(i,N_PLAYER) # Col vector
        fund_mat_list.append(std_basis @ np.transpose(std_basis))
    return fund_mat_list

In [ ]:
'''
The following function prepares terminal condition for the Riccati equation
'''
def term_cond_Riccati(LAP,MOD_c,N_PLAYER,fund_mat_list):
    # Terminal condition
    term_cond_mat_list = [MOD_c * (LAP @ fund_mat_list[i] @ LAP) for i in range(N_PLAYER)]
        
    # Flatten each metrix into a vector and concatenate
    term_cond_vec = np.concatenate([np.reshape(term_cond_mat,(-1,)) for term_cond_mat in term_cond_mat_list])
    return term_cond_vec

In [ ]:
'''
The following function returns the solution to the Riccati equation using methods above
Output: Riccati_F_rec as an (N_PLAYER * N_STEPS) array of matrices (N_PLAYER * N_PLAYER), 
for each player and each time step, record the F^i which is the numerical solution to the Riccati equation
'''
def solve_Riccati_eqn(N_PLAYER,N_STEPS,LAP,MOD_c,MOD_T,MOD_a,MOD_q,MOD_eps):
    # Terminal condition
    fund_mat_list = get_fund_mat_list(N_PLAYER)
    term_cond_vec = term_cond_Riccati(LAP,MOD_c,N_PLAYER,fund_mat_list)

    # Numerically solving matrix-valued ODE system
    F_SOLUTION = scipy.integrate.solve_ivp(Riccati, [MOD_T, 0], term_cond_vec, 
                                       args=(MOD_a,MOD_q,LAP,MOD_eps,N_PLAYER,fund_mat_list),
                                       t_eval = np.linspace(MOD_T,0,N_STEPS),method = 'DOP853')
    print('Riccati solved!')
    # Notice that F_SOLUTION.y is of shape N_PLAYER^3 * N_STEPS
    Riccati_F = F_SOLUTION.y
    
    # The solution is now presented backward in time, so we need to reverse the time
    Riccati_F = np.fliplr(Riccati_F)

    # Record the solution F as a list of matrices
    # For each player at each time step record a matrix F
    Riccati_F_rec = np.empty((N_PLAYER,N_STEPS),'O')

    for col_ind in range(Riccati_F.shape[1]):
        # At a fixed time point
        F_fixed_time = Riccati_F[:,col_ind]
    
        # Split into N_PLAYER arrays
        F_fixed_time_list = np.array_split(F_fixed_time,N_PLAYER)
    
        # Reshape each array in the list
        F_mat_fixed_time_list = [np.reshape(F_array,(N_PLAYER,N_PLAYER)) for F_array in F_fixed_time_list]
    
        # Record it
        for player_ind in range(N_PLAYER):
            Riccati_F_rec[player_ind,col_ind] = F_mat_fixed_time_list[player_ind]
    
    return Riccati_F_rec

In [ ]:
# Call the function to get the exact solution
# At a finer time discretization level
Riccati_F_rec = solve_Riccati_eqn(N_PLAYER,N_STEPS_EXACT,LAP,MOD_c,MOD_T,MOD_a,MOD_q,MOD_eps)

### Construct Exact NE from the Solution

In [ ]:
'''
The following function calculates the exact NE at a fixed time step (at time num_dt * dt)
Input: X_MC_tensor as 2-dim tensor (N_PLAYER * N_SIM), model parameters, solution to the Riccati equation
Output: an array of 2-dim (N_SIM * 1) tensors denoting each player's control
'''
def get_exact_control(num_dt,N_PLAYER,N_SIM,MOD_q,X_MC_tensor,Riccati_F_rec,LAP):
    LAP_tensor = torch.Tensor(LAP).to(device)
    
    # An array of 2-dim tensors (N_SIM * 1)
    ctrl = np.empty(N_PLAYER,'O') 
    
    for player_ind in range(N_PLAYER):
        std_basis = get_std_basis(player_ind,N_PLAYER) # Col vector
        std_basis = torch.Tensor(std_basis).to(device)
        
        # Riccati solution
        F_tensor = torch.Tensor(Riccati_F_rec[player_ind,num_dt]).to(device)

        # Formula for NE based on F from the Riccati equation (1 * N_SIM) tensor
        player_ind_ctrl = (- MOD_q * (torch.transpose(std_basis,0,1) @ LAP_tensor @ X_MC_tensor) \
                           - (torch.transpose(std_basis,0,1) @ F_tensor @ X_MC_tensor))
        
        # Transpose to get (N_SIM * 1) tensor
        ctrl[player_ind] = torch.transpose(player_ind_ctrl,0,1)
    return ctrl

## Analytic Solution on Transitive Graphs when $\varepsilon = q^2$

On vertex-transitive graphs, we have proved that the Markovian NE can be constructed in the following steps when $\varepsilon = q^2$:
1. Solve the matrix-valued ODE $R'(t) = \frac{1}{c}{\text{Tr}} \left[Q'(R(t)) e^{-t(a+q)L}\right]e^{-t(a+q)L},\quad R(0) = 0$ where $Q(X) := \left[\det\left(I+cXL\right)\right]^{\frac{1}{N}}$.
2. Compute $P_t = (a+q)L + R'(T-t)cL[I + R(T-t)cL]^{-1}$.
3. Compute $F^i_t = \frac{1}{\frac{\text{Tr}(P_t) - (a+q)\text{Tr}(L)}{N}}[P_t - (a+q)L]e_ie_i^T [P_t - (a+q)L]$.
4. Compute $\hat{\alpha}^i(t,x) = -qe_i^T L x - e_i^T F^i_t x$.

In [ ]:
'''
The following function implements the derivative of the Q function.
Input: an N by N matrix X
Output: an N by N matrix Q'(X)
'''
def trans_Q_prime(X):
    # Compute Q(X)
    Q = ((np.linalg.det(np.eye(N_PLAYER) + MOD_c * (X @ LAP))) ** (1 / N_PLAYER))
    
    # Q'(X) = Q(X) * c / N * (I + cXL)^{-1} * L
    Q_prime = Q * MOD_c / N_PLAYER * (np.linalg.inv(np.eye(N_PLAYER) + MOD_c * (X @ LAP)) @ LAP)
    
    return Q_prime

In [ ]:
'''
The following function calculates R'(t) above in the ODE.
Input: t as time, R_vec as vec(R) (of length N_PLAYER^2), model parameters
Output: the derivative of R as a vector (of length N_PLAYER^2)
'''
def trans_R_derivative(t,R_vec):
    # Reshape as N * N matrix
    R_reshaped = np.reshape(R_vec,(N_PLAYER,N_PLAYER))
    
    # Matrix exponential
    mat_exp = scipy.linalg.expm(-t * (MOD_a + MOD_q) * LAP)
    
    # Compute Q'(R) as an N * N matrix
    R_prime = 1 / MOD_c * np.matrix.trace(trans_Q_prime(R_reshaped) @ mat_exp) * mat_exp
    
    # Reshape into a vector with N^2 entries
    R_prime_reshaped = np.reshape(R_prime,(-1,))
    
    return R_prime_reshaped

In [ ]:
'''
The following function returns the solution to the ODE above
Output: trans_R_rec as a size N_STEPS array of matrices (N_PLAYER * N_PLAYER), 
for each time step, record the numerical solution to the ODE
'''
def solve_transitive_ODE(N_PLAYER,N_STEPS):
    # Initial condition
    init_cond_vec = np.reshape(np.zeros((N_PLAYER,N_PLAYER)),(-1,))

    # Numerically solving matrix-valued ODE system
    R_SOLUTION = scipy.integrate.solve_ivp(trans_R_derivative, [0, MOD_T], init_cond_vec, 
                                       t_eval = np.linspace(0,MOD_T,N_STEPS),method = 'DOP853')

    # Notice that R_SOLUTION.y is of shape N_PLAYER^2 * N_STEPS
    trans_R = R_SOLUTION.y

    # Record the solution R as a list of matrices
    # For each time step record a matrix R
    trans_R_rec = np.empty(N_STEPS,'O')

    for col_ind in range(trans_R.shape[1]):
        # At a fixed time point
        R_fixed_time = trans_R[:,col_ind]
    
        # Reshape each array in the list
        R_fixed_time_reshaped = np.reshape(R_fixed_time,(N_PLAYER,N_PLAYER))
    
        # Record it
        trans_R_rec[col_ind] = R_fixed_time_reshaped
    
    return trans_R_rec

In [ ]:
'''
This function computes P based on the R solved
Output: trans_P_rec as a size N_STEPS array of matrices (N_PLAYER * N_PLAYER), 
for each time step
'''
def get_trans_P(N_PLAYER,N_STEPS):
    # Numerically solve for R
    trans_R_rec = solve_transitive_ODE(N_PLAYER,N_STEPS)
    
    # Compute P
    trans_P_rec = np.empty(N_STEPS,'O')
    for time_ind in range(N_STEPS):
        # Get R(T - t)
        trans_R_vec_rev = np.flip(trans_R_rec)
        trans_R_rev = trans_R_vec_rev[time_ind]
        
        # Get T - t
        t_eval_rev = np.flip(np.linspace(0,MOD_T,N_STEPS))
        t_rev = t_eval_rev[time_ind]
    
        # Get R'(T-t)
        trans_R_rev_reshaped = np.reshape(trans_R_rev,(-1,))
        R_prime_rev_vec = trans_R_derivative(t_rev,trans_R_rev_reshaped)
        R_prime_rev = np.reshape(R_prime_rev_vec,(N_PLAYER,N_PLAYER))
        
        # Compute P
        trans_P_rec[time_ind] = ((MOD_a + MOD_q) * LAP + MOD_c * (R_prime_rev
                @ LAP @ np.linalg.inv(np.eye(N_PLAYER) + MOD_c * trans_R_rev @ LAP)))
    return trans_P_rec

In [ ]:
'''
This function computes F for each player at each time step based on P
for transitive graphs.
Output: trans_F_rec as a size (N_PLAYER * N_STEPS) array of matrices (N_PLAYER * N_PLAYER), 
for each player at each time step 
'''
def get_trans_F(N_PLAYER,N_STEPS):
    # Get P at each time step
    trans_P_rec = get_trans_P(N_PLAYER,N_STEPS)
    
    # Record F for each player at each time step
    trans_F_rec = np.empty((N_PLAYER,N_STEPS),'O')
    fund_mat_list = get_fund_mat_list(N_PLAYER)
    for time_ind in range(N_STEPS):
        # Get P_t
        trans_P = trans_P_rec[time_ind]
        tau = (np.matrix.trace(trans_P) - (MOD_a + MOD_q) * np.matrix.trace(LAP)) / N_PLAYER
        tmp_mat = trans_P - (MOD_a + MOD_q) * LAP
        
        for ply_ind in range(N_PLAYER):
            trans_F_rec[ply_ind,time_ind] = (1 / tau) * (tmp_mat @ fund_mat_list[ply_ind] @ tmp_mat)
    return trans_F_rec

In [ ]:
# Call the function to get the F for transitive graphs when EPS = q^2
# At a finer time discretization level
if TRANSITIVE:
    trans_F_rec = get_trans_F(N_PLAYER,N_STEPS_EXACT)
    print("Semi-explicit NE for transitive graph calculated!")

### Construct NE on Transitive Graphs

In [ ]:
'''
The following function calculates the NE at a fixed time step (at time num_dt * dt)
for transitive graphs when EPS = q^2
Input: X_MC_tensor as 2-dim tensor (N_PLAYER * N_SIM), model parameters, solution to the Riccati equation
Output: an array of 2-dim (N_SIM * 1) tensors denoting each player's control
'''
def get_trans_NE(num_dt,N_PLAYER,N_SIM,MOD_q,X_MC_tensor,trans_F_rec):
    LAP_tensor = torch.Tensor(LAP).to(device)
    
    # An array of 2-dim tensors (N_SIM * 1)
    ctrl = np.empty(N_PLAYER,'O') 
    
    for player_ind in range(N_PLAYER):
        std_basis = get_std_basis(player_ind,N_PLAYER) # Col vector
        std_basis = torch.Tensor(std_basis).to(device)
        
        # Riccati solution
        F_tensor = torch.Tensor(trans_F_rec[player_ind,num_dt]).to(device)

        # Formula for NE based on F from the Riccati equation (1 * N_SIM) tensor
        player_ind_ctrl = (- MOD_q * (torch.transpose(std_basis,0,1) @ LAP_tensor @ X_MC_tensor) \
                           - (torch.transpose(std_basis,0,1) @ F_tensor @ X_MC_tensor))
        
        # Transpose to get (N_SIM * 1) tensor
        ctrl[player_ind] = torch.transpose(player_ind_ctrl,0,1)
    return ctrl

## Analytic (closed-form) Solution on Complete Graph

The NE $\hat{\alpha}(t,x)$ for such model (on observing current time $t$ and current state vector $x$) on complete graph actually has closed-form solution as provided below.
$$\hat{\alpha}(t,x) = (q + \eta_t)(\overline{x}^{(-i)} - x^i)$$
where $\overline{x}^{(-i)} = \frac{1}{N-1}\sum_{j=1,j\neq i}^N x^j$ is the mean state of all players except player $i$ and $\eta_t$ is a deterministic function in $t$.

The formula of $\eta_t$ is given below.
$$\begin{cases}
        \eta_t = \frac{(c\delta^- - C) -(c\delta^+ - C)e^{(\delta^+ - \delta^-)(T-t)}}{(Ac-\delta^+) + (\delta^- - Ac)e^{(\delta^+ - \delta^-)(T-t)}}\\
        A = \frac{N+1}{N-1}\\
        B = \frac{2N}{N-1}(a+q)\\
        C = -(\varepsilon-q^2)\\
        R = \frac{B^2}{4} -AC\\
        \delta^\pm = -\frac{B}{2}\pm \sqrt{R}
    \end{cases}$$

In [ ]:
'''
Compute eta(t)
Input: time t as a real number and model parameters
Output: eta(t) as a real number
'''
def function_eta(t,N_PLAYER,MOD_a,MOD_sigma,MOD_q,MOD_eps,MOD_c,MOD_T):
    A = (N_PLAYER + 1) / (N_PLAYER - 1)
    B = 2 * N_PLAYER / (N_PLAYER - 1) * (MOD_a + MOD_q)
    C = -(MOD_eps - MOD_q ** 2)
    R = (B ** 2 / 4) - A * C
    delta_plus = -(B / 2) + np.sqrt(R)
    delta_minus = -(B / 2) - np.sqrt(R)
    exp_factor = np.exp((delta_plus - delta_minus) * (MOD_T - t))
    numer = (MOD_c * delta_minus - C) - (MOD_c * delta_plus - C) * exp_factor
    denom = (A * MOD_c - delta_plus) + (delta_minus - A * MOD_c) * exp_factor
    return numer / denom

In [ ]:
'''
Mean reverting term with the mean not including the certain player itself
Input: X_tensor as 2-dim tensor (N_PLAYER * N_SIM), the index of player
Output: a 2-dim column tensor (N_SIM * 1)
'''
def mean_rev_term_all_except_one(X_tensor,player_fixed_ind,N_PLAYER):
    # Get all players' indices except the given one
    except_one_player_ind = list(range(player_fixed_ind)) + list(range(player_fixed_ind + 1,N_PLAYER))
    
    # Maintain the dimension as 2-dim (N_SIM * 1)
    X_tensor_except_one = X_tensor[except_one_player_ind,:]
    mean_tensor = torch.unsqueeze(torch.mean(X_tensor_except_one,dim = 0),1)
    player_fixed_ind_tensor = torch.unsqueeze(X_tensor[player_fixed_ind,:],1)
    return mean_tensor - player_fixed_ind_tensor

### Construct Analytic NE for LQ Game on Complete Graph

In [ ]:
'''
Get all players' controls at a fixed time step from the true NE
Input: X_tensor as 2-dim tensor (N_PLAYER * N_SIM)
Output: an array of 2-dim (N_SIM * 1) tensors denoting each player's control
'''
def get_analytic_NE_complete(X_tensor,num_dt,dt,N_PLAYER,N_SIM,MOD_a,MOD_sigma,MOD_q,MOD_eps,MOD_c,MOD_T):
    ctrl = np.empty(N_PLAYER,'O') 
    # Current time
    t = num_dt * dt
    
    # Under current time, calculate eta(t)
    eta_value = function_eta(t,N_PLAYER,MOD_a,MOD_sigma,MOD_q,MOD_eps,MOD_c,MOD_T)
    for player_ind in range(N_PLAYER):
        ctrl[player_ind] = (MOD_q + eta_value) * mean_rev_term_all_except_one(X_tensor,player_ind,N_PLAYER)
    return ctrl

## An Auxiliary Function

The following function turns the array of 2-dimensional tensors of size $N_{SIM} \times 1$ into 2-dimensional tensors of size $N_{PLAYER} \times N_{SIM}$ to make vectorized calculations easier.

In [ ]:
'''
Converting X_MC from an array (length N_PLPYER) of 2-dim (N_SIM * 1) tensors to a 2-dim (N_PLAYER * N_SIM) tensor
Input: X_MC, model parameters
Output: a 2-dim tensor (N_PLAYER * N_SIM)
'''
def get_two_dim_tensor(X_MC,N_PLAYER,N_sim):
    X_MC_tensor = X_MC[0]
    # The state vector of all players
    for ply_ind in range(1,N_PLAYER):
        X_MC_tensor = torch.cat((X_MC_tensor,X_MC[ply_ind]),1)
    # Now it's N_sim * N_PLAYER shape, do transpose to get N_PLAYER * N_SIM shape
    return torch.transpose(X_MC_tensor,0,1)

## The Test Function

Set up initial condition in test setting. Fix the random seed to ensure that the trajectories plotted has the same initial state.

In [ ]:
'''
Set initial value condition for the state dynamics in test env
Input: model parameters
Output: X_MC updated with initial value conditions
'''
def set_init_val_test(N_PLAYER,N_SIM):
    init_X_MC = np.empty(N_PLAYER,'O')
    '''
    # Generally, we can set x0 = 0.5 * i for player i
    for ply_ind in range(N_PLAYER):
        init_X_MC[ply_ind] = (0.5 * ply_ind * torch.ones((N_SIM,1))).to(device)
    '''
    
    # Uniform initial condition U(-delta,delta)
    torch.manual_seed(TORCH_INIT_SEED)
    for ply_ind in range(N_PLAYER):
        init_X_MC[ply_ind] = (2 * MOD_DELTA * torch.rand(N_SIM,1) - MOD_DELTA * torch.ones(N_SIM,1)).to(device)
    return init_X_MC

Build up a test function to compare the expected cost under DFP solution and the closed-form solution. 

The idea is to simulate SDE under NN approximated and closed-form solution controls with the same BM trajectories.

In [ ]:
'''
Compute the expected cost / state and control process based on given controls of all players
Input: model parameters,
get_ctrl and get_ctrl_flag indicating how to get control, get_ctrl_flag can only take values 'exact' or 
'analytic_complete' or 'semi-explicit'

Parameter N_SIM denotes the number of Monte Carlo iterations in the testing process, which is typically 
much more than the batch size (although they share the same name).

BM_increment flag to indicate if BM_increment is fixed or not, it only takes value 'fixed' or 'not-fixed'
If the flag is fixed, we provide the BM increments as the input and check its dimension.
N_SIM_TEST shall not be too large when BM_increment_flag is 'fixed' since it causes memory overflow.

The arguments Riccati_F_rec and LAP are only used for building up exact solution.

Output: 
the expected cost as a real number if BM_increment_flag is 'not-fixed'
state and control processes of all players as extra output if BM_increment_flag is 'fixed' (for plotting)
'''
def get_exp_cost(ply_FP_ind,
                    MOD_T,NB_LST_LST,MOD_a,MOD_q,MOD_eps,MOD_c,MOD_sigma,N_PLAYER,N_STEPS,
                    get_ctrl,get_ctrl_flag,BM_increment_flag,BM_increment,N_SIM_TEST,Riccati_F_rec,LAP):
    # Check flag values
    assert (BM_increment_flag in ['fixed','not-fixed'] 
            and get_ctrl_flag in ['exact','analytic_complete','semi_explicit']), 'Error in flags!'
    
    # If BM_increment flag is 'fixed', check the input's dimension
    # BM_increment shall be a tensor of dimension N_PLAYER * N_STEPS * N_SIM_TEST
    # Check the number of trajectories fixed (to prevent memory overflow)
    if BM_increment_flag == 'fixed':
        assert (BM_increment.shape == (N_PLAYER,N_STEPS,N_SIM_TEST) and N_SIM_TEST <= 10),'Error! BM increment dimension does not match!'
    
    # Calculate dt
    dt = MOD_T / N_STEPS
    
    # Let X be an array of a torch tensors of size N_SIM so we can do the Monte Carlo once and for all
    # All players' states at a fixed time step
    X_MC = np.empty(N_PLAYER,'O')
    next_X_MC = np.empty(N_PLAYER,'O')
    
    # Record the cumulative running cost for the certain player
    cumu_running_cost_MC = torch.zeros((N_SIM_TEST,1)).to(device)
    
    # Initial condition set for all players
    X_MC = set_init_val_test(N_PLAYER,N_SIM_TEST)
    
    # Record state and control process according to BM_increment flag
    if BM_increment_flag == 'fixed':
        # Record the state and control process
        state_rec = np.zeros((N_PLAYER,N_STEPS,N_SIM_TEST))
        ctrl_rec = np.zeros((N_PLAYER,N_STEPS,N_SIM_TEST))
        
    #-------------------------------------------------------------
    # At each time step
    for num_dt in range(N_STEPS):
        
        # Convert X_MC to a 2-dimensional tensor (N_PLAYER * N_SIM_TEST)
        X_MC_tensor = get_two_dim_tensor(X_MC,N_PLAYER,N_SIM_TEST)
        
        # For general graphs, the exact solution is calculated through numerically solving the Riccati equations
        if get_ctrl_flag == 'exact':
            # Check the shape of Riccati_F_rec (shall match time discretization)
            assert (Riccati_F_rec.shape[1] == N_STEPS), 'Error! Riccati_F_rec shape does not match!'
            ctrl = get_exact_control(num_dt,N_PLAYER,N_SIM_TEST,MOD_q,X_MC_tensor,Riccati_F_rec,LAP)
        # For transitive graphs, the exact solution is calculated through numerically solving the Riccati equations
        elif get_ctrl_flag == 'semi_explicit' and TRANSITIVE:
            # Check the shape of trans_F_rec (shall match time discretization)
            assert (trans_F_rec.shape[1] == N_STEPS), 'Error! Trans_F_rec shape does not match!'
            ctrl = get_trans_NE(num_dt,N_PLAYER,N_SIM_TEST,MOD_q,X_MC_tensor,trans_F_rec)
        # For complete graph, there is analytic solution
        elif get_ctrl_flag == 'analytic_complete':
            # Get the analytic NE
            ctrl = get_analytic_NE_complete(X_MC_tensor,num_dt,dt,N_PLAYER,N_SIM_TEST,MOD_a,MOD_sigma,MOD_q,MOD_eps,MOD_c,MOD_T)
        
        # Calculate the state of all players at the next time step
        for ply_ind in range(N_PLAYER):
            
            # Get this player's control (2-dim tensor (N_SIM * 1))
            ctrl_ply_ind = ctrl[ply_ind]
            
            # Set up the drift term for this player in Euler scheme (2-dim column tensor)
            drift = dynamics_b(X_MC_tensor,ply_ind,NB_LST_LST,MOD_a,ctrl_ply_ind)
            
            # BM increments (depending on the flag)
            if BM_increment_flag == 'not-fixed':
                dW_t = torch.randn((N_SIM_TEST,1)).to(device) * np.sqrt(dt)
            elif BM_increment_flag == 'fixed':
                # Notice that the slicing of BM_increment is 1-dim but dW_t shall be 2-dim
                dW_t = torch.reshape(BM_increment[ply_ind,num_dt,:],(-1,1)).to(device)
                
                # Record the state and control now if necessary (2-dim reshape to 1-dim)
                tmp_state = X_MC[ply_ind].cpu() # Pull back to CPU in order to change to np array
                tmp_ctrl = ctrl[ply_ind].cpu()
                state_rec[ply_ind,num_dt,:] = tmp_state.detach().numpy().reshape(-1)
                ctrl_rec[ply_ind,num_dt,:] = tmp_ctrl.detach().numpy().reshape(-1)
            
            # Vectorized calculations for MC samples, Euler scheme
            next_X_MC[ply_ind] = X_MC[ply_ind] + drift * dt + MOD_sigma * dW_t
        
        # Cumulative running cost for that certain player
        # Notice that we shall multiply by dt since it's an integral
        ctrl_ply_FP_ind = ctrl[ply_FP_ind]
        cumu_running_cost_MC = cumu_running_cost_MC + dt * dynamics_f(X_MC_tensor,
                                        ply_FP_ind,NB_LST_LST,MOD_q,MOD_eps,ctrl_ply_FP_ind)
        
        # Proceed to the next time step
        X_MC = next_X_MC
            
    # Terminal cost for that certain player
    terminal_cost_MC = dynamics_g(X_MC_tensor,ply_FP_ind,NB_LST_LST,MOD_c)
    
    # Calculate expected total cost
    exp_cost = torch.mean((cumu_running_cost_MC + terminal_cost_MC)).detach().item()
    
    # Return different values according to the flag
    if BM_increment_flag == 'not-fixed':
        return exp_cost
    elif BM_increment_flag == 'fixed':
        return exp_cost, state_rec, ctrl_rec

## Numerics to present

## 1. Check the correctness of the semi-explicit NE

### Compute MRE

$$\text{MAE} := \max_{t\in \Delta}\max_{i\in[N]}||\check{F}^i_t - \tilde{F}^i_t||$$
$$\text{MRE} := \max_{t\in \Delta}\max_{i\in[N]}\frac{||\check{F}^i_t - \tilde{F}^i_t||}{||\check{F}^i_t||}$$

In [ ]:
'''
The following function computes the maximum relative error for F.
Input: two size (N_PLAYER * N_STEPS) arrays of matrices (N_PLAYER * N_PLAYER), 
for each player at each time step, one as exact, the other as semi-explicit.
Ouput: MRE
'''
def compute_MAE_MRE(exact_F_rec,semi_F_rec,N_PLAYER,N_STEPS):
    MAE = 0
    MRE = 0
    for ply_ind in range(N_PLAYER):
        for time_ind in range(N_STEPS):
            diff_norm = np.linalg.norm(exact_F_rec[ply_ind,time_ind] - semi_F_rec[ply_ind,time_ind])
            exact_norm = np.linalg.norm(exact_F_rec[ply_ind,time_ind])
            rel_norm = diff_norm / exact_norm
            MAE = np.maximum(MAE,diff_norm)
            MRE = np.maximum(MRE,rel_norm)
    return MAE, MRE

In [ ]:
# A txt file to save the following results
fd = open('numerics.txt','w')

Compute the maximum relative error across all the players and report it in the paper.

In [ ]:
if TRANSITIVE:
    # Calculate MAE, MRE
    MAE, MRE = compute_MAE_MRE(Riccati_F_rec,trans_F_rec,N_PLAYER,N_STEPS_EXACT)
    fd.write('MAE: ' + str(MAE) + '\n')
    fd.write('MRE: ' + str(MRE) + '\n')

In [ ]:
'''
The following function prepares BM increments for two time discretization levels in a consistent way
Input: model parameters
Output: two tensors as the BM increments for coarser and finer level of time discretization respectively
'''
def get_BM_increment(MOD_T,N_STEPS,N_STEPS_EXACT,N_PLAYER,N_FIXED_BM):
    # Calculate dt and organize BM increments w.r.t. N_STEPS_EXACT steps
    exact_dt = MOD_T / N_STEPS_EXACT
    EXACT_BM_increment = torch.randn((N_PLAYER,N_STEPS_EXACT,N_FIXED_BM)).to(device) * np.sqrt(exact_dt)

    # Add up BM increments from finer time discret to get BM increments from coarser time discret
    BM_increment = torch.zeros((N_PLAYER,N_STEPS,N_FIXED_BM)).to(device)
    step_ratio = int(N_STEPS_EXACT / N_STEPS)
    
    for p_ind in range(N_PLAYER):
        for BM_ind in range(N_FIXED_BM):
            for t_ind in range(N_STEPS):
                t_ind_start = t_ind * step_ratio
                t_ind_end = (t_ind + 1) * step_ratio
                BM_increment[p_ind,t_ind,BM_ind] = (torch.sum(EXACT_BM_increment[p_ind,t_ind_start:t_ind_end,BM_ind]))
    
    return BM_increment, EXACT_BM_increment

In [ ]:
# Fix BM trajectory, we have already made sure that N_STEPS_EXACT is a multiple of N_STEPS
BM_increment_flag = 'fixed'

# For reproducibility of plots, fix torch random seed
torch.manual_seed(TORCH_PLOT_SEED)
BM_increment, EXACT_BM_increment = get_BM_increment(MOD_T,N_STEPS,N_STEPS_EXACT,N_PLAYER,N_FIXED_BM)

In [ ]:
# The fixed player now does not matter, just take it as 0 (since we just care about the state and control)
ply_FP_ind = 0

# Exact solution (use different BM increments and finer time steps)
get_ctrl_flag = 'exact'
get_ctrl = get_exact_control
_, Exact_state_rec, Exact_ctrl_rec = get_exp_cost(ply_FP_ind,
                    MOD_T,NB_LST_LST,MOD_a,MOD_q,MOD_eps,MOD_c,MOD_sigma,N_PLAYER,N_STEPS_EXACT,
                    get_ctrl,get_ctrl_flag,BM_increment_flag,EXACT_BM_increment,N_FIXED_BM,Riccati_F_rec,LAP)

# Only plot exact solution at coarser time discret level
step_ratio = int(N_STEPS_EXACT / N_STEPS)
Exact_state_rec = Exact_state_rec[:,::step_ratio,:] # List slicing
Exact_ctrl_rec = Exact_ctrl_rec[:,::step_ratio,:]

# Semi-explicit for transitive if available
if TRANSITIVE:
    get_ctrl_flag = 'semi_explicit'
    get_ctrl = get_trans_NE
    _, Semi_state_rec, Semi_ctrl_rec = get_exp_cost(ply_FP_ind,
                    MOD_T,NB_LST_LST,MOD_a,MOD_q,MOD_eps,MOD_c,MOD_sigma,N_PLAYER,N_STEPS_EXACT,
                    get_ctrl,get_ctrl_flag,BM_increment_flag,EXACT_BM_increment,N_FIXED_BM,Riccati_F_rec,LAP)

    # Only plot exact solution at coarser time discret level
    step_ratio = int(N_STEPS_EXACT / N_STEPS)
    Semi_state_rec = Semi_state_rec[:,::step_ratio,:] # List slicing
    Semi_ctrl_rec = Semi_ctrl_rec[:,::step_ratio,:]

# Analytic solution if available
if graph_name == 'complete':
    get_ctrl_flag = 'analytic_complete'
    get_ctrl = get_analytic_NE_complete
    _, Analytic_state_rec, Analytic_ctrl_rec = get_exp_cost(ply_FP_ind,
                    MOD_T,NB_LST_LST,MOD_a,MOD_q,MOD_eps,MOD_c,MOD_sigma,N_PLAYER,N_STEPS,
                    get_ctrl,get_ctrl_flag,BM_increment_flag,BM_increment,N_FIXED_BM,Riccati_F_rec,LAP)

Plot all players' state and control curves (set N_FIXED_BM as one).

In [ ]:
'''
# The offset for the plot for complete graph
ax1_x_offset = [0,0,0,0,0]
ax1_y_offset = [0.1,-0.2,0.1,-0.2,0.2]
ax2_x_offset = [0,0,0,0,0]
ax2_y_offset = [0.2,-0.25,-0.15,-0.2,0.2]
'''
# The offset for the plot for cycle graph
ax1_x_offset = [0,0,-0.1,0,0]
ax1_y_offset = [0.1,0.2,0.2,-0.2,0.2]
ax2_x_offset = [0,0,-0.2,0,0]
ax2_y_offset = [0.2,-0.25,-0.2,-0.2,0.2]

In [ ]:
# Color list
color_lst = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", 
             "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]

# 1 * 2 subplot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (15, 6))

# Plot those curves for each player in the format we want
np.random.seed(seed = 42)
ply_ind_plot_lst = np.random.choice(N_PLAYER,size = 4,replace = False)
ply_ind_plot_lst.sort() # Ascending order
ply_count = 0
for ply_ind in ply_ind_plot_lst:
    
    # x-axis for plotting (time)
    t_plot = np.linspace(0,MOD_T,N_STEPS)
    
    for fixed_BM_ind in range(1):
        # Fix a color
        color_used = color_lst[ply_count]
            
        # Plot DFP result in dashed line, NTM-DFP in real line, exact result in dots 
        # analytic results in marker 'x' (if available) with the same color
        Exact_state_plot, = ax1.plot(t_plot,Exact_state_rec[ply_ind,:,fixed_BM_ind],
                                       linestyle = '-',color = color_used, label = 'Baseline' if ply_count == 0 else '')
        
        Exact_ctrl_plot, = ax2.plot(t_plot,Exact_ctrl_rec[ply_ind,:,fixed_BM_ind],
                                      linestyle = '-',color = color_used, label = 'Baseline' if ply_count == 0 else '')
            
        # Add text labels (player index) to the curves
        ax1.text(t_plot[10 + 5 * ply_count] + ax1_x_offset[ply_count], Exact_state_rec[ply_ind,10 + 5 * ply_count,fixed_BM_ind] + ax1_y_offset[ply_count], str(ply_ind + 1), 
                     fontsize = 10, color = color_used)
        ax2.text(t_plot[10 + 5 * ply_count] + ax2_x_offset[ply_count], Exact_ctrl_rec[ply_ind,10 + 5 * ply_count,fixed_BM_ind] + ax2_y_offset[ply_count], str(ply_ind + 1), 
                     fontsize = 10, color = color_used)
            
        # Closed-form
        if graph_name == 'complete':
            Analytic_state_plot, = ax1.plot(t_plot,Analytic_state_rec[ply_ind,:,fixed_BM_ind],
                                       linestyle = 'None',marker = '.',color = 'black'
                                               , label = 'Closed-form' if ply_count == 0 else ''
                                               , markersize = 4)
            
            Analytic_ctrl_plot, = ax2.plot(t_plot,Analytic_ctrl_rec[ply_ind,:,fixed_BM_ind],
                                       linestyle = 'None',marker = '.',color = 'black'
                                              , label = 'Closed-form' if ply_count == 0 else ''
                                              , markersize = 4)
        # Semi-explicit
        if TRANSITIVE:
            Semi_state_plot, = ax1.plot(t_plot,Semi_state_rec[ply_ind,:,fixed_BM_ind],
                                       linestyle = 'None',marker = 'o',color = color_used
                                           , label = 'Semi-explicit' if ply_count == 0 else ''
                                           , markerfacecolor = 'none')
            
            Semi_ctrl_plot, = ax2.plot(t_plot,Semi_ctrl_rec[ply_ind,:,fixed_BM_ind],
                                       linestyle = 'None',marker = 'o',color = color_used
                                          , label = 'Semi-explicit' if ply_count == 0 else ''
                                          , markerfacecolor = 'none')
        
    # Update player count
    ply_count += 1

# Axis
ax1.set_ylim([-1, 2])
ax2.set_ylim([-2, 2])
ax1.tick_params(axis='both', which='major', labelsize=14)
ax2.tick_params(axis='both', which='major', labelsize=14)
        
# Set legend
ax1.legend(fontsize = 14)
ax2.legend(fontsize = 14)

# Add labels
ax1.text(0.5, -0.2, '(a)', ha='center', va='center', transform=ax1.transAxes, fontsize=12)
ax2.text(0.5, -0.2, '(b)', ha='center', va='center', transform=ax2.transAxes, fontsize=12)
        
# Other attributes
ax1.set_xlabel(r'Time $t$', fontsize = 16)
ax1.set_ylabel(r'$\hat{X}^i_t$', fontsize = 16)
ax2.set_xlabel(r'Time $t$', fontsize = 16)
ax2.set_ylabel(r'$\hat{\alpha}^i_t$', fontsize = 16)

        
# Distance between subplots
#plt.subplots_adjust(left=0.05,bottom=0.2,right=0.95,top=0.8,wspace=0.4,hspace=0.5)
plt.subplots_adjust(bottom=0.2)
        
# Save the figures
fig.savefig('plots/' + 'All_state_ctrl_curve.jpg', dpi = 600)
plt.close()

In [ ]:
fd.close()

## 2. Show the convergence of FP

The mathematical formulation of FP depends on the notation $F^{i,k}:[0,T]\to \mathbb{S}^{N\times N}$, which is the $F$ for player $i$ at FP stage $k$.
It is determined by the following matrix-valued Riccati equation with given terminal condition
$$ \dot{F}^{i,k+1}_t - F^{i,k+1}_te_ie_i^T F^{i,k+1}_t + (\varepsilon-q^2)Le_ie_i^T L - (a+q)LF^{i,k+1}_t - (a+q)F^{i,k+1}_tL \\
    - F^{i,k+1}_t\sum_{j\neq i} e_je_j^T F^{j,k}_t - \sum_{j\neq i} F^{j,k}_te_je_j^T F^{i,k+1}_t = 0$$
$$F^{i,k+1}_T = cLe_ie_i^TL$$
we use the method in scipy package to numerically solve the coupled ODE system given $F^{1,k},...,F^{N,k}$ in order to get $F^{1,k+1},...,F^{N,k+1}$.

The FP procedure comes to a stop when FP provides a solution that is close enough to the last iteration, i.e., $\max_{t\in\Delta}\max_i \lVert F^{i,k}_t - F^{i,k-1}_t\rVert < tol$ for some error tolerance $tol$.

In [ ]:
'''
The following function calculates the derivative of F^{i,k+1}, flattened as a vector and concatenated.
Input: t as time, F as vec(F^{0,k}),...,vec(F^{N-1,k}) concatenated together (of length N_PLAYER^3), model parameters,
the F matrices in the last FP stage
Output: the derivative of F^{i,k+1} (of length N_PLAYER^3)
'''
def FP_Riccati(t,F_vec,MOD_a,MOD_q,LAP,MOD_eps,N_PLAYER,N_STEPS,fund_mat_list,last_FP_F_rec):
    # Select the correct time slice of past F according to current time t
    time_ind = int(np.round(t / N_STEPS))
    last_FP_F_list = last_FP_F_rec[:,time_ind]
    
    # Cut vector y into N_PLAYER pieces and reshape into matrices
    F_piece_list = np.array_split(F_vec,N_PLAYER)
    F_mat_list = [np.reshape(F_piece,(N_PLAYER,N_PLAYER)) for F_piece in F_piece_list]
    
    # Calculate derivative
    F_prime_mat_list = list()
    for i in range(N_PLAYER):
        F_prime_mat = np.zeros((N_PLAYER,N_PLAYER))
        # Riccati equation above
        for j in range(N_PLAYER):
            if j == i:
                continue
            # Only for j not equal to i
            F_prime_mat = F_prime_mat + ((F_mat_list[i] @ fund_mat_list[j] @ last_FP_F_list[j]) 
                                         + (last_FP_F_list[j] @ fund_mat_list[j] @ F_mat_list[i]))
        # EPS - q^2 term
        F_prime_mat = F_prime_mat - (MOD_eps - MOD_q ** 2) * (LAP @ fund_mat_list[i] @ LAP)
        
        # F^{i,k+1}e_ie_i^TF^{i,k+1} term
        F_prime_mat = F_prime_mat + F_mat_list[i] @ fund_mat_list[i] @ F_mat_list[i]
        
        # two (a + q) terms
        F_prime_mat = F_prime_mat + (MOD_a + MOD_q) * (LAP @ F_mat_list[i] + F_mat_list[i] @ LAP)
        
        # Append
        F_prime_mat_list.append(F_prime_mat)
    
    # Flatten back into a vector and concatenate
    F_prime_vec = np.concatenate([np.reshape(F_prime_mat,(-1,)) for F_prime_mat in F_prime_mat_list])
    
    return F_prime_vec

In [ ]:
'''
The following function prepares terminal condition for the FP Riccati equation
'''
def term_cond_FP_Riccati(LAP,MOD_c,N_PLAYER,fund_mat_list):
    # Terminal condition
    term_cond_mat_list = [MOD_c * (LAP @ fund_mat_list[i] @ LAP) for i in range(N_PLAYER)]
        
    # Flatten each metrix into a vector and concatenate
    term_cond_vec = np.concatenate([np.reshape(term_cond_mat,(-1,)) for term_cond_mat in term_cond_mat_list])
    return term_cond_vec

In [ ]:
'''
The following function initializes F for FP procedure
'''
def init_FP_Riccati(N_PLAYER,N_STEPS):
    # Init start as a list of zero matrices
    init_mat_rec = np.empty((N_PLAYER,N_STEPS),'O')
    for ply_ind in range(N_PLAYER):
        for t in range(N_STEPS):
            init_mat_rec[ply_ind,t] = np.zeros((N_PLAYER,N_PLAYER))
            #init_mat_rec[ply_ind,t] = np.eye(N_PLAYER)
    return init_mat_rec

In [ ]:
'''
The following function returns the solution to the Riccati equation using methods above
Output: Record the whole procedure of FP, return FP_history_F_rec as a list of (N_PLAYER * N_STEPS) array of matrices (N_PLAYER * N_PLAYER), 
for each player and each time step.
'''
def solve_FP_Riccati(N_PLAYER,N_STEPS,LAP,MOD_c,MOD_T,MOD_a,MOD_q,MOD_eps,exact_F_rec):
    # Terminal condition
    fund_mat_list = get_fund_mat_list(N_PLAYER)
    term_cond_vec = term_cond_FP_Riccati(LAP,MOD_c,N_PLAYER,fund_mat_list)
    
    # Starts FP with zero matrices
    last_FP_F_rec = init_FP_Riccati(N_PLAYER,N_STEPS)
    
    # Record the whole history of FP to see how F changes
    FP_history_F_rec = list()
    error_list = list()
    
    # Start FP
    FP_count = 0
    while 1:
        
        # Record the FP history
        FP_history_F_rec.append(last_FP_F_rec)

        # Numerically solving recursive matrix-valued ODE system
        F_SOLUTION = scipy.integrate.solve_ivp(FP_Riccati, [MOD_T, 0], term_cond_vec, 
                                       args=(MOD_a,MOD_q,LAP,MOD_eps,N_PLAYER,N_STEPS,fund_mat_list,last_FP_F_rec),
                                       t_eval = np.linspace(MOD_T,0,N_STEPS),method = 'DOP853')

        # Notice that F_SOLUTION.y is of shape N_PLAYER^3 * N_STEPS
        Riccati_F = F_SOLUTION.y
    
        # The solution is now presented backward in time, so we need to reverse the time
        Riccati_F = np.fliplr(Riccati_F)

        # Record the solution F as a list of matrices
        # For each player at each time step record a matrix F
        Riccati_F_rec = np.empty((N_PLAYER,N_STEPS),'O')

        for col_ind in range(Riccati_F.shape[1]):
            # At a fixed time point
            F_fixed_time = Riccati_F[:,col_ind]
    
            # Split into N_PLAYER arrays
            F_fixed_time_list = np.array_split(F_fixed_time,N_PLAYER)
    
            # Reshape each array in the list
            F_mat_fixed_time_list = [np.reshape(F_array,(N_PLAYER,N_PLAYER)) for F_array in F_fixed_time_list]
    
            # Record it
            for player_ind in range(N_PLAYER):
                Riccati_F_rec[player_ind,col_ind] = F_mat_fixed_time_list[player_ind]
        
        # Compute MAE
        MAE, _ = compute_MAE_MRE(Riccati_F_rec,last_FP_F_rec,N_PLAYER,N_STEPS_EXACT)
        error_list.append(MAE)
                
        # Update the last_FP_F_list and enter the next iteration
        last_FP_F_rec = Riccati_F_rec
        FP_count += 1
        
        if FP_count >= N_FP_ROUND:
            break
    
    return FP_history_F_rec, error_list

# Output the results for different graphs at once

In [ ]:
fd = open('FP_numerics.txt','w')

# Color list
color_lst = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", 
             "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]

In [ ]:
fig, ax = plt.subplots(figsize = (10, 6))

In [ ]:
'''
# The list for producing numerics
graph_result_lst = [‘RSG','RSG','RSG','Ramanujan','Ramanujan','Ramanujan']
legend_lst = [r'Random sparse RSG$^{0.125}_{50}$'] * 3 + [r'Ramanujan RMJ$^6_{50}$'] * 3
'''


# The list for producing plots
graph_result_lst = ['complete','cycle','star','complete bipartite','RSG','Ramanujan']

legend_lst = [r'Complete $K_{50}$',r'Cycle $C_{50}$',r'Star $S_{50}$',r'Complete bipartite $K_{25,25}$',
              r'Random sparse RSG$^{0.125}_{50}$',r'Ramanujan RMJ$^6_{50}$']


graph_ind = 0

for graph_name in graph_result_lst:
    
    LAP = set_graph_LAP(graph_name,graph_name_lst)
    
    Riccati_F_rec = solve_Riccati_eqn(N_PLAYER,N_STEPS_EXACT,LAP,MOD_c,MOD_T,MOD_a,MOD_q,MOD_eps)
    
    FP_history_F_rec, error_list = solve_FP_Riccati(N_PLAYER,N_STEPS_EXACT,LAP,MOD_c,MOD_T,MOD_a,MOD_q,MOD_eps,exact_F_rec = Riccati_F_rec)
    
    # Print norm of Laplacian
    L_norm = np.linalg.norm(LAP,2)
    print(str(L_norm))
    
    # Record numerics
    fd.write('Graph: ' + graph_name + '\n')
    fd.write('N_round: ' + str(len(FP_history_F_rec)) + '\n')
    FP_final_result = FP_history_F_rec[-1]
    MAE, MRE = compute_MAE_MRE(Riccati_F_rec,FP_final_result,N_PLAYER,N_STEPS_EXACT)
    fd.write('MAE: ' + str(MAE) + '\n')
    fd.write('MRE: ' + str(MRE) + '\n')
    
    # Plot
    t_plot = np.arange(1,N_FP_ROUND + 1)
    log_error_plot, = ax.plot(t_plot,np.log(error_list),color = color_lst[graph_ind], label = legend_lst[graph_ind])
    
    # Estimate the slope and print
    lr_X = np.reshape(t_plot,(-1,1))
    lr_Y = np.log(error_list)
    model = LinearRegression()
    model.fit(lr_X,lr_Y)
    fd.write('C_G estimate: ' + str(model.coef_) + '\n')
    
    graph_ind += 1

In [ ]:
check = '\u030C'
ax.set_xlabel('FP stage index', fontsize = 16)
ax.set_ylabel('log-MAE', fontsize = 16)

# Axis
ax.tick_params(axis='both', which='major', labelsize=14)   
# Set legend
ax.legend(fontsize = 14)

fig.savefig('plots/FP_graph_error.jpg', dpi = 600)
plt.close()

## End of Code

In [ ]:
# Close the txt file
fd.close()

In [ ]:
# End of code
print('END OF CODE!!!')